# OSHA ITA Case Detail Data: Starter Notebook

**Courses:** ENVH 312 (Parts 1 to 4) and ENVH 460/560 (all parts)

In this notebook you will explore individual workplace injuries and illnesses that
employers reported to OSHA through the Injury Tracking Application (ITA).
Read the module [README](../README.md) and [data dictionary](../data_dictionary.md) first.

**How to use this notebook**
- Run each code cell in order (Shift + Enter).
- Answer the questions in the cells marked **Your answer**. Double-click a cell to edit it.
- Cells marked *ENVH 460/560* are required for graduate students and optional for ENVH 312.

## Part 0. Load the data

By default this notebook uses the **sample** (a few thousand random cases per year),
which is committed to the repository. To use the **full** data, run
`scripts/fetch_data.py` first and set `USE_FULL_DATA = True`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

USE_FULL_DATA = False

MODULE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = MODULE_DIR / "data" / "raw"
SAMPLE_PATH = MODULE_DIR / "data" / "sample" / "ita_case_detail_sample.csv"
SAMPLE_URL = ("https://raw.githubusercontent.com/eddiekasner/occupational-safety-health-datasets/"
              "main/modules/us-osha-ita-case-detail/data/sample/ita_case_detail_sample.csv")


def load_ita():
    if USE_FULL_DATA:
        files = sorted(p for p in RAW_DIR.glob("*") if p.suffix.lower() in {".csv", ".zip"})
        if not files:
            raise FileNotFoundError(f"No files in {RAW_DIR}. Run scripts/fetch_data.py first.")
        frames = []
        for path in files:
            part = pd.read_csv(path, dtype=str, low_memory=False, encoding_errors="replace")
            part["source_file"] = path.name
            frames.append(part)
        df = pd.concat(frames, ignore_index=True)
    else:
        source = SAMPLE_PATH if SAMPLE_PATH.exists() else SAMPLE_URL  # URL works in Google Colab
        df = pd.read_csv(source, dtype=str, low_memory=False)
    df.columns = df.columns.str.strip().str.lower()
    return df


df = load_ita()
print(f"{len(df):,} cases, {df.shape[1]} columns")

In [ ]:
def pick(*names):
    """Return the first of these column names that exists (names can differ between years)."""
    for name in names:
        if name in df.columns:
            return name
    print(f"None of {names} found. Check df.columns against the data dictionary.")
    return None


OUTCOME = pick("incident_outcome")
DAYS_AWAY = pick("dafw_num_away")
TYPE = pick("type_of_incident")
NAICS = pick("naics_code")
STATE = pick("state")
SOC = pick("soc_description", "soc_code")
YEAR = pick("year_of_filing", "year_filing_for", "source_file")

outcome_labels = {"1": "Death", "2": "Days away from work", "3": "Job transfer or restriction",
                  "4": "Other recordable case"}
type_labels = {"1": "Injury", "2": "Skin disorder", "3": "Respiratory condition",
               "4": "Poisoning", "5": "Hearing loss", "6": "All other illnesses"}

## Part 1. Get oriented

Before analyzing anything, find out what one row represents and what the columns contain.

In [ ]:
print(list(df.columns))
df.head(3).T

In [ ]:
if YEAR:
    print(df[YEAR].value_counts().sort_index())

**Questions**
1. What does one row in this dataset represent?
2. Pick three columns you did not expect to see. What are they, and why might OSHA collect them?
3. Which employers are required to submit this data? Who is *not* represented?

**Your answer:**

## Part 2. How serious are these cases?

In [ ]:
outcomes = df[OUTCOME].map(outcome_labels).fillna("Unknown/missing")
counts = outcomes.value_counts()
print((counts / counts.sum() * 100).round(1).astype(str) + " %")
counts.plot.barh(title="Cases by most serious outcome")
plt.xlabel("Number of cases")
plt.tight_layout()

In [ ]:
types = df[TYPE].map(type_labels).fillna("Unknown/missing")
types.value_counts()

**Questions**
1. What share of cases involved days away from work? Job transfer or restriction?
2. Illnesses make up a small share of cases. Give two reasons why work-related
   illnesses are probably under-recorded on OSHA logs.

**Your answer:**

## Part 3. Days away from work

In [ ]:
days = pd.to_numeric(df[DAYS_AWAY], errors="coerce")
days = days[days > 0]
print(days.describe().round(1))
print(f"Mean: {days.mean():.1f}   Median: {days.median():.1f}")
days.clip(upper=180).plot.hist(bins=36, title="Days away from work (capped at 180)")
plt.xlabel("Days")
plt.tight_layout()

**Questions**
1. Compare the mean and median. What does the difference tell you about the shape of the distribution?
2. OSHA recordkeeping caps days away at 180 for a single case. How might that affect your analysis?

**Your answer:**

## Part 4. Who and where?

NAICS codes are hierarchical. The first two digits give the industry sector
(for example 23 = construction, 31 to 33 = manufacturing, 48 to 49 = transportation and warehousing,
62 = health care and social assistance).

In [ ]:
df["naics_2"] = df[NAICS].str[:2]
df["naics_2"].value_counts().head(10)

In [ ]:
df[SOC].value_counts().head(15)

In [ ]:
if STATE:
    print(df[STATE].value_counts().head(10))

**Questions**
1. Which sectors and occupations account for the most cases?
2. Does having the most cases mean an industry is the *most dangerous*? What would you need to
   know to compare risk between industries? (Hint: see "incidence rate" in the data dictionary.)

**Your answer:**

## Part 5 (ENVH 460/560). Reading the narratives

Form 301 asks the employer to describe what happened in their own words. Find the narrative
columns and search them for a hazard of your choice.

In [ ]:
narrative_cols = [c for c in df.columns if "nar" in c or "description" in c or "what_happened" in c]
print(narrative_cols)

In [ ]:
KEYWORD = "heat"   # try: ladder, forklift, lifting, needle, slip, chemical, violence

text = df[narrative_cols].fillna("").agg(" ".join, axis=1).str.lower()
hits = df[text.str.contains(KEYWORD, regex=False)]
print(f"{len(hits):,} cases mention '{KEYWORD}' ({len(hits) / len(df):.1%} of cases)")
hits[narrative_cols].head(10)

**Questions**
1. Read 10 matching narratives. How many actually describe the hazard you searched for? What
   does that tell you about keyword searches as a surveillance method?
2. Pick one hazard. Using the hierarchy of controls, propose one control that would have
   prevented two of the cases you read.

**Your answer:**

## Part 6 (ENVH 460/560). From counts to rates

To compare industries fairly you need a denominator. The ITA **300A summary** file has
`total_hours_worked` for each establishment and links to this file through `establishment_id`.

The DART rate is:

$$\text{DART rate} = \frac{\text{cases with days away, restriction, or transfer} \times 200{,}000}{\text{total hours worked}}$$

**Questions**
1. Sketch (in words or pseudocode) how you would compute a DART rate by NAICS sector using
   both files. What join key would you use? What could go wrong if an establishment
   submitted a 300A but no case detail, or the reverse?
2. This dataset only includes establishments with 100+ employees in designated industries.
   How might that selection affect rates compared with BLS's national SOII estimates?
3. Propose a research question that this dataset could answer and that aggregated BLS
   tables could not.

**Your answer:**